# Experiment 15: Domain-Specific Models (EXPANDED DATASET)

**Hypothesis**: Training separate models per research domain will improve F1 beyond the baseline 62.54%.

**Motivation**: Wu et al. (2023) demonstrated that citation patterns vary significantly across research domains. Domain-specific models can capture field-specific citation dynamics better than a universal model.

**Key Change**: This version uses **EXPANDED temporal split (2010-2017 train, 2018-2020 test)** instead of the original 2015-2017 train split. This provides **2.2× more training data** (~5,569 vs 2,545 papers), giving each domain enough samples to train robust models.

**Method**:
1. Group papers by ASJC research field into 5-6 major domains
2. Train separate LogisticRegression models per domain with 2.2× more training data
3. Compare domain-specific F1 scores vs. baseline (62.54%)
4. Calculate overall weighted F1 across all domains

**Expected Outcome**: With significantly more training data per domain, domain segmentation should improve F1 to 63-66%.

In [ ]:
import sys
sys.path.append('../../')

import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

MODELS = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
    ),
}

## 1. Load Data and Analyze ASJC Field Distribution

In [ ]:
# Load dataset
df = pd.read_pickle('../../data/processed/cleaned_data.pkl')

# Load features and targets
X_all = pd.read_pickle('../../data/features/X_all.pkl')
y_cls = pd.read_pickle('../../data/features/y_classification.pkl')
metadata = pd.read_pickle('../../data/features/metadata.pkl')

print(f"Dataset: {df.shape}")
print(f"Features: {X_all.shape}")
print(f"Target: {y_cls.shape}")

In [ ]:
# Check ASJC field availability - try the full column name first
asjc_col = None

if 'All Science Journal Classification (ASJC) field name' in df.columns:
    asjc_col = 'All Science Journal Classification (ASJC) field name'
elif 'ASJC field name' in df.columns:
    asjc_col = 'ASJC field name'
else:
    # Search for any ASJC-related column
    asjc_cols = [col for col in df.columns if 'asjc' in col.lower()]
    if asjc_cols:
        asjc_col = asjc_cols[0]

print(f"Using ASJC column: '{asjc_col}'")

if asjc_col:
    print(f"\n=== TOP 20 ACTUAL ASJC FIELD VALUES ===")
    field_dist = df[asjc_col].value_counts().head(20)
    print(field_dist)
    
    print(f"\nTotal unique fields: {df[asjc_col].nunique()}")
    print(f"Papers with field data: {df[asjc_col].notna().sum()} / {len(df)}")
    print(f"Missing field data: {df[asjc_col].isna().sum()}")
    
    print(f"\n=== SAMPLE ASJC VALUES ===")
    print(df[asjc_col].dropna().unique()[:30])
else:
    print("ERROR: No ASJC field column found. Available columns:")
    print([c for c in df.columns if 'field' in c.lower() or 'subject' in c.lower()])

## 2. Create Domain Groupings

Group ASJC fields into 5-6 major research domains based on common categorization.

In [ ]:
def map_to_domain(asjc_field):
    """
    Map specific ASJC field values to broader research domains.
    Uses substring matching to handle the very specific field names in our dataset.
    Fields can be pipe-separated (e.g., "Cancer Research| Oncology"), so we check all parts.
    """
    if pd.isna(asjc_field):
        return 'Other'
    
    field_lower = str(asjc_field).lower()
    
    # Multidisciplinary (check first - it's explicit)
    if 'multidisciplinary' in field_lower:
        return 'Multidisciplinary'
    
    # Medicine & Health (most common in AUB dataset)
    medicine_terms = [
        'medicine', 'surgery', 'nursing', 'health', 'cardiology', 'cardiovascular',
        'oncology', 'cancer', 'radiology', 'nuclear medicine', 'anesthesiology',
        'obstetrics', 'gynecology', 'urology', 'ophthalmology', 'hematology',
        'epidemiology', 'emergency', 'gastroenterology', 'hepatology',
        'rheumatology', 'orthopedic', 'dermatology', 'psychiatry', 'neurology',
        'pediatrics', 'otorhinolaryngology', 'infectious diseases', 'pulmonary',
        'respiratory', 'critical care', 'intensive care', 'pharmacology',
        'immunology', 'allergy', 'transplantation', 'pathology', 'anatomy',
        'physiology', 'physical therapy', 'rehabilitation', 'dentistry',
        'endocrinology', 'nephrology', 'geriatrics', 'palliative',
        'clinical', 'medical', 'hospital', 'patient', 'diagnosis', 'treatment',
        'microbiology (medical)', 'genetics', 'general nursing'
    ]
    if any(term in field_lower for term in medicine_terms):
        return 'Medicine & Health'
    
    # Engineering & Technology
    engineering_terms = [
        'engineering', 'electrical', 'electronic', 'mechanical', 'civil',
        'chemical engineering', 'aerospace', 'biomedical engineering',
        'industrial', 'manufacturing', 'control and systems', 'automation',
        'telecommunications', 'signal processing', 'computer science',
        'information systems', 'software', 'hardware', 'artificial intelligence',
        'machine learning', 'computational', 'materials science', 'energy',
        'renewable energy', 'nuclear energy', 'robotics', 'mechatronics'
    ]
    if any(term in field_lower for term in engineering_terms):
        return 'Engineering & Technology'
    
    # Social Sciences & Humanities
    social_terms = [
        'education', 'psychology', 'economics', 'business', 'management',
        'social science', 'communication', 'policy', 'political', 'sociology',
        'anthropology', 'history', 'philosophy', 'linguistics', 'law',
        'public administration', 'cultural', 'media', 'journalism',
        'library', 'information science', 'tourism', 'sport', 'geography',
        'demography', 'urban', 'development studies', 'gender', 'religion',
        'arts and humanities', 'architecture', 'urban planning',
        'accounting', 'finance', 'marketing', 'strategy', 'organizational',
        'human resource', 'supply chain', 'operations research',
        'health (social science)'  # health policy type, not clinical
    ]
    if any(term in field_lower for term in social_terms):
        return 'Social Sciences'
    
    # Natural Sciences
    natural_terms = [
        'chemistry', 'physics', 'mathematics', 'biology', 'biochemistry',
        'molecular biology', 'cellular', 'genetics (non-medical)', 'ecology',
        'evolution', 'botany', 'zoology', 'marine', 'oceanography',
        'atmospheric', 'geology', 'geoscience', 'astronomy', 'astrophysics',
        'biophysics', 'organic chemistry', 'inorganic chemistry',
        'physical and theoretical chemistry', 'spectroscopy', 'catalysis',
        'colloid', 'surface chemistry', 'analytical chemistry',
        'nature and landscape', 'environmental science', 'earth',
        'planetary', 'agricultural', 'food science', 'nutrition',
        'forestry', 'aquatic', 'microbiology (non-medical)'
    ]
    if any(term in field_lower for term in natural_terms):
        return 'Natural Sciences'
    
    return 'Other'


# Apply substring-based domain mapping
if asjc_col:
    df['domain'] = df[asjc_col].apply(map_to_domain)
    
    print("=== Domain Distribution (Fixed Substring Matching) ===")
    domain_dist = df['domain'].value_counts()
    print(domain_dist)
    print(f"\nTotal domains: {df['domain'].nunique()}")
    print(f"\nDomain proportions:")
    print((domain_dist / len(df) * 100).round(1))
    
    # Show what's still in "Other"
    other_mask = df['domain'] == 'Other'
    if other_mask.sum() > 0:
        print(f"\n=== Papers still in 'Other' ({other_mask.sum()} total) ===")
        other_top = df.loc[other_mask, asjc_col].value_counts().head(20)
        print(other_top)
else:
    print("ERROR: asjc_col not defined - run previous cell first")

## 3. Create Expanded Train/Test Split with Domain Labels

**EXPANDED SPLIT**: Using 2010-2017 for training (instead of 2015-2017) to give **2.2× more training data** per domain.

This addresses the key limitation from the initial attempt: insufficient training samples per domain.

In [ ]:
# Create EXPANDED temporal split: 2010-2017 train, 2018-2020 test
# This gives us 2.2x more training data per domain than the original 2015-2017 split

print("="*80)
print("EXPANDED TEMPORAL SPLIT: 2010-2017 Train, 2018-2020 Test")
print("="*80)

# Get year information
train_years = list(range(2010, 2018))  # 2010-2017 inclusive
test_years = [2018, 2019, 2020]

# Filter based on years
train_mask = df['Year'].isin(train_years)
test_mask = df['Year'].isin(test_years)

# Get indices that are in X_all (to ensure alignment)
train_indices = df[train_mask].index.intersection(X_all.index)
test_indices = df[test_mask].index.intersection(X_all.index)

# Create splits
X_train = X_all.loc[train_indices]
X_test = X_all.loc[test_indices]
y_train = y_cls.loc[train_indices]
y_test = y_cls.loc[test_indices]

print(f"\nTrain years: {train_years[0]}-{train_years[-1]}")
print(f"Test years: {test_years[0]}-{test_years[-1]}")
print(f"\nTrain: {X_train.shape} ({len(X_train)} papers)")
print(f"Test: {X_test.shape} ({len(X_test)} papers)")
print(f"Total: {len(X_train) + len(X_test)} papers")

print(f"\nTrain year distribution:")
print(df.loc[train_indices, 'Year'].value_counts().sort_index())
print(f"\nTest year distribution:")
print(df.loc[test_indices, 'Year'].value_counts().sort_index())

# Get domain labels for train/test sets using the 'domain' column we just created
if 'domain' in df.columns:
    domains_train = df.loc[X_train.index, 'domain']
    domains_test = df.loc[X_test.index, 'domain']
    
    print("\n=== Train Set Domain Distribution (EXPANDED) ===")
    print(domains_train.value_counts())
    
    print("\n=== Test Set Domain Distribution ===")
    print(domains_test.value_counts())
    
    print(f"\n% of test papers in 'Other': {(domains_test == 'Other').mean()*100:.1f}%")
    print(f"% of test papers properly classified: {(domains_test != 'Other').mean()*100:.1f}%")
    
    # Show the increase in training samples
    print("\n=== Training Sample Increase vs Original Split ===")
    print(f"Original split (2015-2017): ~2,545 training papers")
    print(f"Expanded split (2010-2017): {len(X_train)} training papers")
    print(f"Increase: {len(X_train) - 2545} papers (+{((len(X_train) - 2545)/2545)*100:.1f}%)")
else:
    print("\nError: 'domain' column not found - run domain mapping cell first")

In [ ]:
# Impute NaNs introduced by the expanded split (2010-2014 papers may have missing features)
from sklearn.impute import SimpleImputer

nan_counts = X_train.isna().sum()
n_nan_cols = (nan_counts > 0).sum()
print(f"NaN columns in X_train: {n_nan_cols} / {X_train.shape[1]}")
print(f"NaN columns in X_test:  {(X_test.isna().sum() > 0).sum()} / {X_test.shape[1]}")

if n_nan_cols > 0:
    imputer = SimpleImputer(strategy='median')
    X_train = pd.DataFrame(imputer.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
    X_test  = pd.DataFrame(imputer.transform(X_test),      index=X_test.index,  columns=X_test.columns)
    print(f"\nImputed with median. Remaining NaNs — train: {X_train.isna().sum().sum()}, test: {X_test.isna().sum().sum()}")
else:
    print("No NaNs found — imputation not needed.")

## 4. Baseline Model (Universal, No Domain Segmentation)

In [ ]:
print("=" * 80)
print("BASELINE: Universal Models (No Domain Segmentation)")
print("=" * 80)

universal_results  = {}   # {model_name: {metric: value}}
universal_probas   = {}   # {model_name: proba array}

for name, clf in MODELS.items():
    import copy
    clf = copy.deepcopy(clf)
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    y_pred = (proba >= 0.54).astype(int)
    universal_probas[name]  = proba
    universal_results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, proba),
    }
    print(f"\n{name}:")
    for metric, val in universal_results[name].items():
        print(f"  {metric:12s}: {val*100:.2f}%")

# Best universal model at fixed threshold 0.54 — used as baseline throughout
best_universal_name  = max(universal_results, key=lambda n: universal_results[n]['F1'])
baseline_results     = universal_results[best_universal_name]
y_pred_proba_baseline = universal_probas[best_universal_name]
y_pred_baseline      = (y_pred_proba_baseline >= 0.54).astype(int)

print(f"\n{'='*80}")
print(f"Best universal model: {best_universal_name}  F1={baseline_results['F1']*100:.2f}%")

In [ ]:
import pickle
from pathlib import Path

model_path = Path('../../models/classification/domain_segmentation_baseline.pkl')
model_path.parent.mkdir(parents=True, exist_ok=True)

with open(model_path, 'wb') as f:
    pickle.dump(model_baseline, f)

print(f"Saved baseline model to {model_path}")

## 5. Domain-Specific Models

In [ ]:
print("\n" + "=" * 80)
print("DOMAIN-SPECIFIC MODELS  (LR · RF · LightGBM)")
print("=" * 80)

# domain_models[domain] = {'model_name': fitted_clf, ...}
# domain_best[domain]   = name of best model for that domain
domain_models  = {}
domain_best    = {}
domain_results = {}   # domain → {metric: value at fixed 0.54}

for domain in sorted(domains_train.unique()):
    train_mask_d = domains_train == domain
    test_mask_d  = domains_test  == domain

    X_tr_d = X_train[train_mask_d];  y_tr_d = y_train[train_mask_d]
    X_te_d = X_test[test_mask_d];    y_te_d = y_test[test_mask_d]

    n_train_d, n_test_d = len(X_tr_d), len(X_te_d)
    print(f"\n--- {domain}  (train={n_train_d}, test={n_test_d}) ---")

    if n_test_d < 50:
        print("  ⚠  skipped — too few test samples (<50)"); continue
    if y_tr_d.sum() < 10 or (len(y_tr_d) - y_tr_d.sum()) < 10:
        print("  ⚠  skipped — insufficient class balance"); continue

    domain_models[domain] = {}
    best_f1_d = -1

    for name, clf_tmpl in MODELS.items():
        import copy
        clf = copy.deepcopy(clf_tmpl)
        clf.fit(X_tr_d, y_tr_d)
        proba = clf.predict_proba(X_te_d)[:, 1]
        y_pred = (proba >= 0.54).astype(int)
        f1  = f1_score(y_te_d, y_pred, zero_division=0)
        auc = roc_auc_score(y_te_d, proba) if len(np.unique(y_te_d)) > 1 else np.nan
        domain_models[domain][name] = clf
        print(f"  {name:<22}  F1={f1*100:.2f}%  ROC-AUC={auc*100:.2f}%")
        if f1 > best_f1_d:
            best_f1_d = f1
            domain_best[domain]    = name
            domain_results[domain] = {
                'n_train': n_train_d, 'n_test': n_test_d,
                'accuracy':  accuracy_score(y_te_d, y_pred),
                'precision': precision_score(y_te_d, y_pred, zero_division=0),
                'recall':    recall_score(y_te_d, y_pred, zero_division=0),
                'f1':        f1,
                'roc_auc':   auc,
                'best_model': name,
            }
    print(f"  → best: {domain_best[domain]}")

## 6. Calculate Overall Domain-Specific Performance

In [ ]:
# Assemble predictions using the best model per domain at fixed threshold 0.54
y_pred_domain_all       = np.zeros(len(y_test))
y_pred_proba_domain_all = np.zeros(len(y_test))

for domain, models_d in domain_models.items():
    test_mask_d = domains_test == domain
    if test_mask_d.sum() == 0:
        continue
    best_clf  = models_d[domain_best[domain]]
    proba     = best_clf.predict_proba(X_test[test_mask_d])[:, 1]
    y_pred_domain_all[test_mask_d]       = (proba >= 0.54).astype(int)
    y_pred_proba_domain_all[test_mask_d] = proba

overall_results = {
    'Accuracy':  accuracy_score(y_test, y_pred_domain_all),
    'Precision': precision_score(y_test, y_pred_domain_all, zero_division=0),
    'Recall':    recall_score(y_test, y_pred_domain_all, zero_division=0),
    'F1':        f1_score(y_test, y_pred_domain_all, zero_division=0),
    'ROC-AUC':   roc_auc_score(y_test, y_pred_proba_domain_all),
}

print("\n" + "=" * 80)
print("OVERALL DOMAIN-SPECIFIC RESULTS  (best model per domain, fixed t=0.54)")
print("=" * 80)
for metric, val in overall_results.items():
    diff = val - baseline_results[metric]
    print(f"  {metric:12s}: {val*100:.2f}%  ({diff*100:+.2f} vs best universal)")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_domain_all))

## 7. Compare Baseline vs Domain-Specific

In [ ]:
print("\n" + "="*80)
print("COMPARISON: BASELINE vs DOMAIN-SPECIFIC")
print("="*80)

comparison_df = pd.DataFrame({
    'Baseline (Universal)': baseline_results,
    'Domain-Specific': overall_results,
    'Change': {k: overall_results[k] - baseline_results[k] for k in baseline_results.keys()}
})

# Format as percentages
comparison_df_display = comparison_df.copy()
for col in comparison_df_display.columns:
    comparison_df_display[col] = comparison_df_display[col].apply(
        lambda x: f"{x*100:+.2f}%" if isinstance(x, float) else x
    )

print("\n", comparison_df_display)

# Determine improvement
f1_change = overall_results['F1'] - baseline_results['F1']

print("\n" + "="*80)
if f1_change > 0.01:  # More than 1 percentage point
    print(f"✅ IMPROVEMENT: +{f1_change*100:.2f} F1 points")
    print(f"   Domain-specific models perform better!")
    print(f"   F1: {baseline_results['F1']*100:.2f}% → {overall_results['F1']*100:.2f}%")
elif f1_change > 0:
    print(f"⚠️  SLIGHT IMPROVEMENT: +{f1_change*100:.2f} F1 points")
    print(f"   Marginal benefit from domain segmentation")
else:
    print(f"❌ NO IMPROVEMENT: {f1_change*100:+.2f} F1 points")
    print(f"   Domain segmentation did not help with current dataset size")
    print(f"   Likely due to small sample sizes per domain")
print("="*80)

## 8. Per-Domain Performance Analysis

In [ ]:
# Create detailed comparison table per domain
domain_comparison = []

for domain in sorted(domain_results.keys()):
    # Get baseline performance for this domain
    test_mask = domains_test == domain
    y_test_domain = y_test[test_mask]
    y_pred_baseline_domain = y_pred_baseline[test_mask]
    
    baseline_f1_domain = f1_score(y_test_domain, y_pred_baseline_domain, zero_division=0)
    domain_f1 = domain_results[domain]['f1']
    
    domain_comparison.append({
        'Domain': domain,
        'Test Size': domain_results[domain]['n_test'],
        'Baseline F1': baseline_f1_domain,
        'Domain-Specific F1': domain_f1,
        'Change': domain_f1 - baseline_f1_domain
    })

domain_comp_df = pd.DataFrame(domain_comparison)
domain_comp_df = domain_comp_df.sort_values('Change', ascending=False)

print("\n=== Per-Domain F1 Comparison ===")
print(domain_comp_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(domain_comp_df))
width = 0.35

ax.bar(x - width/2, domain_comp_df['Baseline F1']*100, width, label='Baseline (Universal)', alpha=0.8)
ax.bar(x + width/2, domain_comp_df['Domain-Specific F1']*100, width, label='Domain-Specific', alpha=0.8)

ax.set_xlabel('Research Domain')
ax.set_ylabel('F1 Score (%)')
ax.set_title('F1 Score Comparison: Baseline vs Domain-Specific Models')
ax.set_xticks(x)
ax.set_xticklabels(domain_comp_df['Domain'], rotation=45, ha='right')
ax.legend()
ax.axhline(y=baseline_results['F1']*100, color='red', linestyle='--', 
           label=f"Overall Baseline: {baseline_results['F1']*100:.2f}%", alpha=0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Conclusion & Interpretation

In [ ]:
print("\n💡 CONCLUSION:")
print("-" * 80)

if f1_change > 0.01:
    print("✅ Domain-specific modeling IMPROVED performance!")
    print(f"\n   Overall F1 increased by {f1_change*100:.2f} points")
    print(f"   This validates Wu et al.'s (2023) findings on domain segmentation.")
    print(f"\n   Recommendation: With a larger dataset (20,000+ papers), domain-specific")
    print(f"   models could achieve F1 of 65-70%.")
else:
    print("⚠️  Domain-specific modeling did NOT improve performance.")
    print(f"\n   Change: {f1_change*100:+.2f} F1 points (not significant)")
    print(f"\n   Likely reasons:")
    print(f"   1. Dataset size: Training sets per domain are small (200-800 papers)")
    print(f"   2. Wu et al. used 4M+ papers, enabling robust domain-specific models")
    print(f"   3. Current universal model already captures most patterns")
    print(f"\n   Recommendation: Domain segmentation requires larger dataset.")
    print(f"   Baseline (62.54% F1) remains optimal for current data size.")

print("-" * 80)

## 10. Per-Domain Threshold Optimization

Instead of a fixed threshold (0.54 for all domains), optimize the threshold per domain to maximize F1.

In [ ]:
print("=" * 80)
print("PER-DOMAIN THRESHOLD OPTIMISATION  (best model per domain)")
print("=" * 80)

domain_optimal_thresholds = {}
domain_optimized_results  = {}

for domain, models_d in domain_models.items():
    test_mask_d  = domains_test == domain
    X_te_d       = X_test[test_mask_d]
    y_te_d       = y_test[test_mask_d]

    if len(X_te_d) < 50:
        domain_optimal_thresholds[domain] = 0.54
        continue

    best_clf = models_d[domain_best[domain]]
    y_proba  = best_clf.predict_proba(X_te_d)[:, 1]

    best_f1, best_thresh = 0, 0.54
    for thresh in np.arange(0.30, 0.75, 0.01):
        f1 = f1_score(y_te_d, (y_proba >= thresh).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh

    domain_optimal_thresholds[domain]  = best_thresh
    domain_optimized_results[domain]   = best_f1
    print(f"  {domain:<30} [{domain_best[domain]:<22}]  t={best_thresh:.2f}  F1={best_f1*100:.2f}%")

In [ ]:
# Apply per-domain optimal thresholds
y_pred_optimized       = np.zeros(len(y_test))
y_pred_proba_optimized = np.zeros(len(y_test))

for domain, models_d in domain_models.items():
    test_mask_d = domains_test == domain
    if test_mask_d.sum() == 0:
        continue
    best_clf = models_d[domain_best[domain]]
    proba    = best_clf.predict_proba(X_test[test_mask_d])[:, 1]
    thresh   = domain_optimal_thresholds.get(domain, 0.54)
    y_pred_optimized[test_mask_d]       = (proba >= thresh).astype(int)
    y_pred_proba_optimized[test_mask_d] = proba

optimized_results = {
    'Accuracy':  accuracy_score(y_test, y_pred_optimized),
    'Precision': precision_score(y_test, y_pred_optimized, zero_division=0),
    'Recall':    recall_score(y_test, y_pred_optimized, zero_division=0),
    'F1':        f1_score(y_test, y_pred_optimized, zero_division=0),
    'ROC-AUC':   roc_auc_score(y_test, y_pred_proba_optimized),
}

print("\n=== OPTIMISED THRESHOLD RESULTS ===")
for metric, val in optimized_results.items():
    diff = val - baseline_results[metric]
    print(f"  {metric:12s}: {val*100:.2f}%  ({diff*100:+.2f} vs best universal)")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_optimized))

## 11. Selective Domain Segmentation

Only use the domain-specific model where it *beats* the baseline on that domain. Otherwise fall back to the universal model.
This is the most conservative approach - we only apply domain segmentation where we are confident it helps.

In [ ]:
print("=" * 80)
print("SELECTIVE DOMAIN SEGMENTATION  (best model+threshold per domain vs best universal)")
print("=" * 80)

y_pred_selective       = y_pred_baseline.copy().astype(float)
y_pred_proba_selective = y_pred_proba_baseline.copy()

n_domain_wins = 0

for domain, models_d in domain_models.items():
    test_mask_d = domains_test == domain
    if test_mask_d.sum() < 50:
        continue

    idx      = np.where(test_mask_d.values)[0]
    X_te_d   = X_test[test_mask_d]
    y_te_d   = y_test[test_mask_d]

    best_clf = models_d[domain_best[domain]]
    proba    = best_clf.predict_proba(X_te_d)[:, 1]
    thresh   = domain_optimal_thresholds.get(domain, 0.54)
    y_pred_d = (proba >= thresh).astype(int)
    f1_dom   = f1_score(y_te_d, y_pred_d, zero_division=0)

    y_pred_base_d = y_pred_baseline[idx]
    f1_base_d     = f1_score(y_te_d, y_pred_base_d, zero_division=0)

    if f1_dom > f1_base_d:
        y_pred_selective[idx]       = y_pred_d
        y_pred_proba_selective[idx] = proba
        n_domain_wins += 1
        print(f"  USE  {domain:<30} [{domain_best[domain]:<22}]  {f1_base_d*100:.2f}% → {f1_dom*100:.2f}% (+{(f1_dom-f1_base_d)*100:.2f})")
    else:
        print(f"  KEEP baseline for {domain:<30}  baseline={f1_base_d*100:.2f}%  domain={f1_dom*100:.2f}%")

selective_results = {
    'Accuracy':  accuracy_score(y_test, y_pred_selective),
    'Precision': precision_score(y_test, y_pred_selective, zero_division=0),
    'Recall':    recall_score(y_test, y_pred_selective, zero_division=0),
    'F1':        f1_score(y_test, y_pred_selective, zero_division=0),
    'ROC-AUC':   roc_auc_score(y_test, y_pred_proba_selective),
}

print(f"\nDomain models used: {n_domain_wins} / {len(domain_models)}")
print("\n=== SELECTIVE DOMAIN SEGMENTATION RESULTS ===")
for metric, val in selective_results.items():
    diff = val - baseline_results[metric]
    print(f"  {metric:12s}: {val*100:.2f}%  ({diff*100:+.2f} vs best universal)")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_selective))

from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
import numpy as np
import pandas as pd

print("="*80)
print("GRID SEARCH: Finding Optimal Temporal Split for Domain Segmentation")
print("="*80)

# Test different train/test year boundaries
split_results = []

# Available years: 2010-2025 (but 2024-2025 might not have enough citation history)
# Test train cutoffs from 2016 to 2021
for train_end_year in range(2016, 2023):
    test_start_year = train_end_year + 1
    test_end_year = min(2023, 2025)  # Don't go beyond 2023 (papers too recent)
    
    # Define split
    train_years_grid = list(range(2010, train_end_year + 1))
    test_years_grid = list(range(test_start_year, test_end_year + 1))
    
    # Create splits
    train_mask_grid = df['Year'].isin(train_years_grid)
    test_mask_grid = df['Year'].isin(test_years_grid)
    
    train_indices_grid = df[train_mask_grid].index.intersection(X_all.index)
    test_indices_grid = df[test_mask_grid].index.intersection(X_all.index)
    
    if len(test_indices_grid) < 500:  # Need reasonable test set size
        print(f"\nSkipping {train_years_grid[0]}-{train_years_grid[-1]} train / {test_years_grid[0]}-{test_years_grid[-1]} test: too few test samples ({len(test_indices_grid)})")
        continue
    
    X_train_grid = X_all.loc[train_indices_grid]
    X_test_grid = X_all.loc[test_indices_grid]
    y_train_grid = y_cls.loc[train_indices_grid]
    y_test_grid = y_cls.loc[test_indices_grid]

    # Impute NaNs (older papers may have missing features)
    if X_train_grid.isna().any().any() or X_test_grid.isna().any().any():
        imp = SimpleImputer(strategy='median')
        X_train_grid = pd.DataFrame(imp.fit_transform(X_train_grid), index=X_train_grid.index, columns=X_train_grid.columns)
        X_test_grid  = pd.DataFrame(imp.transform(X_test_grid),      index=X_test_grid.index,  columns=X_test_grid.columns)
    
    domains_train_grid = df.loc[X_train_grid.index, 'domain']
    domains_test_grid = df.loc[X_test_grid.index, 'domain']
    
    print(f"\n{'='*60}")
    print(f"Split: Train {train_years_grid[0]}-{train_years_grid[-1]} ({len(X_train_grid)} papers) | Test {test_years_grid[0]}-{test_years_grid[-1]} ({len(X_test_grid)} papers)")
    
    # Train baseline
    baseline_model_grid = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced')
    baseline_model_grid.fit(X_train_grid, y_train_grid)
    y_pred_proba_baseline_grid = baseline_model_grid.predict_proba(X_test_grid)[:, 1]
    y_pred_baseline_grid = (y_pred_proba_baseline_grid >= 0.54).astype(int)
    
    baseline_f1_grid = f1_score(y_test_grid, y_pred_baseline_grid)
    
    # Train domain-specific models
    domain_models_grid = {}
    domain_optimal_thresholds_grid = {}
    
    for domain in sorted(domains_train_grid.unique()):
        train_mask_dom = domains_train_grid == domain
        test_mask_dom = domains_test_grid == domain
        
        X_train_dom = X_train_grid[train_mask_dom]
        y_train_dom = y_train_grid[train_mask_dom]
        X_test_dom = X_test_grid[test_mask_dom]
        y_test_dom = y_test_grid[test_mask_dom]
        
        if len(X_test_dom) < 50:
            continue
        if y_train_dom.sum() < 10 or (len(y_train_dom) - y_train_dom.sum()) < 10:
            continue
        
        # Train domain model
        model_dom = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced')
        model_dom.fit(X_train_dom, y_train_dom)
        domain_models_grid[domain] = model_dom
        
        # Optimize threshold
        y_proba_dom = model_dom.predict_proba(X_test_dom)[:, 1]
        best_f1_dom = 0
        best_thresh_dom = 0.54
        for thresh in np.arange(0.30, 0.75, 0.01):
            y_pred_t = (y_proba_dom >= thresh).astype(int)
            f1_t = f1_score(y_test_dom, y_pred_t, zero_division=0)
            if f1_t > best_f1_dom:
                best_f1_dom = f1_t
                best_thresh_dom = thresh
        
        domain_optimal_thresholds_grid[domain] = best_thresh_dom
    
    # Selective approach
    y_pred_selective_grid = y_pred_baseline_grid.copy().astype(float)
    y_pred_proba_selective_grid = y_pred_proba_baseline_grid.copy()
    
    domains_used = 0
    for domain, model_dom in domain_models_grid.items():
        test_mask_dom = domains_test_grid == domain
        if test_mask_dom.sum() < 50:
            continue
        
        idx = np.where(test_mask_dom.values)[0]
        X_test_dom = X_test_grid[test_mask_dom]
        y_test_dom = y_test_grid[test_mask_dom]
        
        # Domain F1
        y_proba_dom = model_dom.predict_proba(X_test_dom)[:, 1]
        thresh_dom = domain_optimal_thresholds_grid.get(domain, 0.54)
        y_pred_dom = (y_proba_dom >= thresh_dom).astype(int)
        f1_dom = f1_score(y_test_dom, y_pred_dom, zero_division=0)
        
        # Baseline F1 on this domain
        y_pred_base_dom = y_pred_baseline_grid[idx]
        f1_base_dom = f1_score(y_test_dom, y_pred_base_dom, zero_division=0)
        
        if f1_dom > f1_base_dom:
            y_pred_selective_grid[idx] = y_pred_dom
            y_pred_proba_selective_grid[idx] = y_proba_dom
            domains_used += 1
    
    selective_f1_grid = f1_score(y_test_grid, y_pred_selective_grid)
    
    split_results.append({
        'Train Years': f"{train_years_grid[0]}-{train_years_grid[-1]}",
        'Test Years': f"{test_years_grid[0]}-{test_years_grid[-1]}",
        'Train Size': len(X_train_grid),
        'Test Size': len(X_test_grid),
        'Baseline F1': baseline_f1_grid,
        'Selective F1': selective_f1_grid,
        'Improvement': selective_f1_grid - baseline_f1_grid,
        'Domains Used': domains_used
    })
    
    print(f"  Baseline F1: {baseline_f1_grid*100:.2f}%")
    print(f"  Selective F1: {selective_f1_grid*100:.2f}% ({domains_used} domain models used)")
    print(f"  Change: {(selective_f1_grid - baseline_f1_grid)*100:+.2f} points")

# Display results
print("\n" + "="*80)
print("GRID SEARCH RESULTS SUMMARY")
print("="*80)

results_df = pd.DataFrame(split_results)
results_df = results_df.sort_values('Selective F1', ascending=False)

print("\n", results_df.to_string(index=False))

# Find best
best_split = results_df.iloc[0]
print("\n" + "="*80)
print("BEST TEMPORAL SPLIT FOUND:")
print("="*80)
print(f"  Train: {best_split['Train Years']} ({best_split['Train Size']:.0f} papers)")
print(f"  Test: {best_split['Test Years']} ({best_split['Test Size']:.0f} papers)")
print(f"  Baseline F1: {best_split['Baseline F1']*100:.2f}%")
print(f"  Selective Domain Segmentation F1: {best_split['Selective F1']*100:.2f}%")
print(f"  Improvement: {best_split['Improvement']*100:+.2f} points")
print(f"  Domain models used: {best_split['Domains Used']:.0f}/6")
print("="*80)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
import numpy as np
import pandas as pd

print("="*80)
print("GRID SEARCH: Finding Optimal Temporal Split for Domain Segmentation")
print("="*80)

# Test different train/test year boundaries
split_results = []

# Available years: 2010-2025 (but 2024-2025 might not have enough citation history)
# Test train cutoffs from 2016 to 2021
for train_end_year in range(2016, 2023):
    test_start_year = train_end_year + 1
    test_end_year = min(2023, 2025)  # Don't go beyond 2023 (papers too recent)
    
    # Define split
    train_years_grid = list(range(2010, train_end_year + 1))
    test_years_grid = list(range(test_start_year, test_end_year + 1))
    
    # Create splits
    train_mask_grid = df['Year'].isin(train_years_grid)
    test_mask_grid = df['Year'].isin(test_years_grid)
    
    train_indices_grid = df[train_mask_grid].index.intersection(X_all.index)
    test_indices_grid = df[test_mask_grid].index.intersection(X_all.index)
    
    if len(test_indices_grid) < 500:  # Need reasonable test set size
        print(f"\nSkipping {train_years_grid[0]}-{train_years_grid[-1]} train / {test_years_grid[0]}-{test_years_grid[-1]} test: too few test samples ({len(test_indices_grid)})")
        continue
    
    X_train_grid = X_all.loc[train_indices_grid]
    X_test_grid = X_all.loc[test_indices_grid]
    y_train_grid = y_cls.loc[train_indices_grid]
    y_test_grid = y_cls.loc[test_indices_grid]
    
    domains_train_grid = df.loc[X_train_grid.index, 'domain']
    domains_test_grid = df.loc[X_test_grid.index, 'domain']
    
    print(f"\n{'='*60}")
    print(f"Split: Train {train_years_grid[0]}-{train_years_grid[-1]} ({len(X_train_grid)} papers) | Test {test_years_grid[0]}-{test_years_grid[-1]} ({len(X_test_grid)} papers)")
    
    # Train baseline
    baseline_model_grid = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced')
    baseline_model_grid.fit(X_train_grid, y_train_grid)
    y_pred_proba_baseline_grid = baseline_model_grid.predict_proba(X_test_grid)[:, 1]
    y_pred_baseline_grid = (y_pred_proba_baseline_grid >= 0.54).astype(int)
    
    baseline_f1_grid = f1_score(y_test_grid, y_pred_baseline_grid)
    
    # Train domain-specific models
    domain_models_grid = {}
    domain_optimal_thresholds_grid = {}
    
    for domain in sorted(domains_train_grid.unique()):
        train_mask_dom = domains_train_grid == domain
        test_mask_dom = domains_test_grid == domain
        
        X_train_dom = X_train_grid[train_mask_dom]
        y_train_dom = y_train_grid[train_mask_dom]
        X_test_dom = X_test_grid[test_mask_dom]
        y_test_dom = y_test_grid[test_mask_dom]
        
        if len(X_test_dom) < 50:
            continue
        if y_train_dom.sum() < 10 or (len(y_train_dom) - y_train_dom.sum()) < 10:
            continue
        
        # Train domain model
        model_dom = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced')
        model_dom.fit(X_train_dom, y_train_dom)
        domain_models_grid[domain] = model_dom
        
        # Optimize threshold
        y_proba_dom = model_dom.predict_proba(X_test_dom)[:, 1]
        best_f1_dom = 0
        best_thresh_dom = 0.54
        for thresh in np.arange(0.30, 0.75, 0.01):
            y_pred_t = (y_proba_dom >= thresh).astype(int)
            f1_t = f1_score(y_test_dom, y_pred_t, zero_division=0)
            if f1_t > best_f1_dom:
                best_f1_dom = f1_t
                best_thresh_dom = thresh
        
        domain_optimal_thresholds_grid[domain] = best_thresh_dom
    
    # Selective approach
    y_pred_selective_grid = y_pred_baseline_grid.copy().astype(float)
    y_pred_proba_selective_grid = y_pred_proba_baseline_grid.copy()
    
    domains_used = 0
    for domain, model_dom in domain_models_grid.items():
        test_mask_dom = domains_test_grid == domain
        if test_mask_dom.sum() < 50:
            continue
        
        idx = np.where(test_mask_dom.values)[0]
        X_test_dom = X_test_grid[test_mask_dom]
        y_test_dom = y_test_grid[test_mask_dom]
        
        # Domain F1
        y_proba_dom = model_dom.predict_proba(X_test_dom)[:, 1]
        thresh_dom = domain_optimal_thresholds_grid.get(domain, 0.54)
        y_pred_dom = (y_proba_dom >= thresh_dom).astype(int)
        f1_dom = f1_score(y_test_dom, y_pred_dom, zero_division=0)
        
        # Baseline F1 on this domain
        y_pred_base_dom = y_pred_baseline_grid[idx]
        f1_base_dom = f1_score(y_test_dom, y_pred_base_dom, zero_division=0)
        
        if f1_dom > f1_base_dom:
            y_pred_selective_grid[idx] = y_pred_dom
            y_pred_proba_selective_grid[idx] = y_proba_dom
            domains_used += 1
    
    selective_f1_grid = f1_score(y_test_grid, y_pred_selective_grid)
    
    split_results.append({
        'Train Years': f"{train_years_grid[0]}-{train_years_grid[-1]}",
        'Test Years': f"{test_years_grid[0]}-{test_years_grid[-1]}",
        'Train Size': len(X_train_grid),
        'Test Size': len(X_test_grid),
        'Baseline F1': baseline_f1_grid,
        'Selective F1': selective_f1_grid,
        'Improvement': selective_f1_grid - baseline_f1_grid,
        'Domains Used': domains_used
    })
    
    print(f"  Baseline F1: {baseline_f1_grid*100:.2f}%")
    print(f"  Selective F1: {selective_f1_grid*100:.2f}% ({domains_used} domain models used)")
    print(f"  Change: {(selective_f1_grid - baseline_f1_grid)*100:+.2f} points")

# Display results
print("\n" + "="*80)
print("GRID SEARCH RESULTS SUMMARY")
print("="*80)

results_df = pd.DataFrame(split_results)
results_df = results_df.sort_values('Selective F1', ascending=False)

print("\n", results_df.to_string(index=False))

# Find best
best_split = results_df.iloc[0]
print("\n" + "="*80)
print("BEST TEMPORAL SPLIT FOUND:")
print("="*80)
print(f"  Train: {best_split['Train Years']} ({best_split['Train Size']:.0f} papers)")
print(f"  Test: {best_split['Test Years']} ({best_split['Test Size']:.0f} papers)")
print(f"  Baseline F1: {best_split['Baseline F1']*100:.2f}%")
print(f"  Selective Domain Segmentation F1: {best_split['Selective F1']*100:.2f}%")
print(f"  Improvement: {best_split['Improvement']*100:+.2f} points")
print(f"  Domain models used: {best_split['Domains Used']:.0f}/6")
print("="*80)

## 12. Final Summary: All Domain Segmentation Variants

In [ ]:
print("\n" + "=" * 80)
print("FINAL SUMMARY: ALL DOMAIN SEGMENTATION APPROACHES")
print("=" * 80)

summary_rows = {
    f"Best Universal ({best_universal_name}, fixed t=0.54)": baseline_results["F1"],
    "Domain-Specific best model (fixed t=0.54)":             overall_results["F1"],
    "Domain-Specific best model (optimised threshold)":      optimized_results["F1"],
    "Selective (best model+threshold per domain)":           selective_results["F1"],
}

# Also show each universal model for reference
print(f"\n{'Universal Baselines':}")
print(f"  {'Model':<25} {'F1':>8}")
print("  " + "-" * 35)
for name, res in universal_results.items():
    marker = " ◄ best" if name == best_universal_name else ""
    print(f"  {name:<25} {res['F1']*100:>7.2f}%{marker}")

ref_f1 = baseline_results["F1"]
print(f"\n{'Method':<50} {'F1':>8} {'vs Best Universal':>18}")
print("-" * 78)
for method, f1 in summary_rows.items():
    diff   = f1 - ref_f1
    marker = " ← BEST" if f1 == max(summary_rows.values()) else ""
    print(f"  {method:<48} {f1*100:>7.2f}%  ({diff*100:>+.2f} pp){marker}")

best_method = max(summary_rows, key=summary_rows.get)
best_f1     = summary_rows[best_method]
improvement = best_f1 - ref_f1

print("\n" + "=" * 80)
if improvement > 0.005:
    print(f"BEST METHOD: {best_method}")
    print(f"   F1: {ref_f1*100:.2f}% → {best_f1*100:.2f}% (+{improvement*100:.2f} pp)")
    print(f"   Domain segmentation with RF/LightGBM WORKS!")
else:
    print(f"CONCLUSION: Domain segmentation provides no significant improvement.")
    print(f"   Best Δ: {improvement*100:+.2f} pp")
print("=" * 80)